# NB_hospital_daily_summary

Builds `hcp_databricks_workspace.gold.hospital_daily_summary` per the STTM (Silver to Gold, v1.0): one row per hospital per day with lab test activity, driven by `silver.lab_results` grouped by hospital_id and test_date, with LEFT JOINs to hospitals and calendar plus a LAG() day-over-day trend. Load type: FULL rebuild.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hcp_databricks_workspace.gold

In [0]:
%sql
CREATE OR REPLACE TABLE hcp_databricks_workspace.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volumes, abnormal/critical rates and day-over-day trend (STTM silver to gold v1.0)'
AS
WITH daily AS (
  -- Driving table: lab_results grouped by hospital and test date
  SELECT
    hospital_id,
    test_date                                  AS summary_date,
    COUNT(*)                                   AS total_tests,
    COUNT(DISTINCT patient_id)                 AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                  AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count
  FROM hcp_databricks_workspace.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_trend AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  t.hospital_id                                                             AS hospital_id,
  h.hospital_name                                                           AS hospital_name,
  t.summary_date                                                            AS summary_date,
  c.day_name                                                                AS day_name,
  c.is_weekend                                                              AS is_weekend,
  CAST(t.total_tests AS INT)                                                AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                     AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                        AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                        AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                        AS critical_test_count,
  CAST(round(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(round(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                       AS prev_day_total_tests,
  CASE
    WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
    ELSE CAST(round((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                       AS test_volume_change_pct
FROM with_trend t
LEFT JOIN hcp_databricks_workspace.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN hcp_databricks_workspace.silver.calendar c
  ON c.datekey = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT)   -- calendar key column is `datekey` (INT yyyyMMdd)

In [0]:
%sql
-- Grain and reconciliation checks against silver
SELECT
  count(*)                                                        AS gold_rows,
  count(DISTINCT hospital_id, summary_date)                       AS distinct_hospital_days,
  sum(total_tests)                                                AS gold_total_tests,
  (SELECT count(*) FROM hcp_databricks_workspace.silver.lab_results) AS silver_total_tests,
  sum(abnormal_test_count)                                        AS abnormal_tests,
  sum(critical_test_count)                                        AS critical_tests,
  count_if(hospital_name IS NULL)                                 AS missing_hospital_name,
  count_if(day_name IS NULL)                                      AS missing_calendar_match,
  count_if(prev_day_total_tests IS NULL)                          AS first_day_rows,
  count(DISTINCT hospital_id)                                     AS hospitals,
  min(summary_date)                                               AS min_date,
  max(summary_date)                                               AS max_date
FROM hcp_databricks_workspace.gold.hospital_daily_summary

In [0]:
%sql
SELECT * FROM hcp_databricks_workspace.gold.hospital_daily_summary ORDER BY hospital_id, summary_date LIMIT 20